# S09: deploy an authenticated local API

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cybertide-solutions/ai-platform-architect-labs/blob/main/notebooks/08_deployment.ipynb)

**Start:** run the first code cell. It automatically downloads the supporting files in a fresh Colab runtime. No ZIP upload, local installation or pasted setup code is required.

70-minute lab. Exercise a real HTTP boundary with server-bound identities. The standard-library server is a single-process teaching deployment, not an internet-facing production server.

In [ ]:
# Automatic lab setup: run this cell first. No ZIP upload or manual clone is needed.
from pathlib import Path
import os, sys, json, tempfile, subprocess, shutil

def prepare_lab_files(colab_home=Path('/content')):
    required = ('engine/__init__.py', 'engine/knowledge.py', 'data/policies.json', 'data/orders.json')
    def ready(path):
        return all((path / name).is_file() for name in required)
    try:
        import google.colab
        in_colab = True
    except ImportError:
        in_colab = False
    target = colab_home / 'ai-platform-architect-labs'
    candidates = ([target] if in_colab else []) + [Path.cwd(), Path.cwd().parent]
    root = next((path for path in candidates if ready(path)), None)
    if root is None and in_colab:
        if target.exists():
            raise RuntimeError('An incomplete course folder already exists. Save your work, then disconnect and delete this Colab runtime and reopen the notebook.')
        print('Downloading the course code and data from GitHub...')
        colab_home.mkdir(parents=True, exist_ok=True)
        staging = Path(tempfile.mkdtemp(prefix='course-download-', dir=colab_home))
        try:
            result = subprocess.run(
                ['git', 'clone', '--quiet', '--depth', '1', '--branch', 'main',
                 'https://github.com/cybertide-solutions/ai-platform-architect-labs.git', str(staging / 'repo')],
                capture_output=True, text=True, timeout=120)
            if result.returncode != 0 or not ready(staging / 'repo'):
                raise RuntimeError('Course download failed. Check the runtime internet connection and rerun this cell. Your existing work was not changed.')
            (staging / 'repo').rename(target)
            root = target
        except subprocess.TimeoutExpired:
            raise RuntimeError('Course download timed out. Rerun this cell to try again.') from None
        finally:
            shutil.rmtree(staging, ignore_errors=True)
    if root is None:
        raise RuntimeError('Local Jupyter: open this notebook from the extracted course folder. In Colab, use the README Open in Colab button.')
    root = root.resolve()
    os.chdir(root)
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    print('Lab files ready. Continue with the cells below.')
    return root

ROOT = prepare_lab_files()
# End automatic setup
from engine import read_data
from engine.identity import BUYER, FINANCE, BEACON
from engine.knowledge import Index, evidence_errors
from engine.purchasing import Purchases, query_errors
from engine.platform import Platform
from engine.provider import ModelClient
# Deliberately opt in. No credential is required for software rehearsal.
# Colab only: enable after saving these names in Colab Secrets and granting access.
USE_COLAB_SECRETS=False
if USE_COLAB_SECRETS:
    from google.colab import userdata
    for name in ['CHAT_URL','CHAT_MODEL','CHAT_KEY','EMBED_URL','EMBED_MODEL','EMBED_KEY']:
        os.environ[name]=userdata.get(name)
    os.environ['LIVE_MODE']='1'
LIVE=os.environ.get('LIVE_MODE')=='1'
client=ModelClient.environment() if LIVE else None
print('LIVE MODEL' if LIVE else 'SOFTWARE REHEARSAL: no generated AI answers or AI quality score')

**0–15 min | Inspect the boundary.** Three distinct generated credentials bind to three server-side application identities. Clients can send a question; they cannot select a tenant or application in the payload. The server defaults to loopback. Live mode disables the explicit query shortcut used for software rehearsal.

In [ ]:
import subprocess,secrets,socket,time,urllib.request,urllib.error
with socket.socket() as sock:
    sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
env=dict(os.environ,LAB_PORT=str(port),LAB_BIND='127.0.0.1',LIVE_MODE='0')
for name in ['POLICY_TOKEN','SPEND_TOKEN','BEACON_TOKEN']:env[name]=secrets.token_urlsafe(24)
process=subprocess.Popen([sys.executable,'-m','engine.server'],env=env,stdout=subprocess.DEVNULL,stderr=subprocess.PIPE)
base=f'http://127.0.0.1:{port}'
for attempt in range(50):
    try:
        with urllib.request.urlopen(base+'/health',timeout=1) as r: print(json.load(r))
        break
    except urllib.error.URLError:time.sleep(.1)
else:
    process.terminate();raise RuntimeError('Local server did not become ready; check port/process setup.')
def request(token,body):
    req=urllib.request.Request(base+'/v1/ask',data=json.dumps(body).encode(),
        headers={'Content-Type':'application/json','Authorization':'Bearer '+token})
    try:
        with urllib.request.urlopen(req,timeout=10) as r:return r.status,json.load(r)
    except urllib.error.HTTPError as r:return r.code,json.load(r)

**15–35 min | Observe the deployed behavior.** Exercise authorised spend, invalid credentials and attempted tenant override. Each result comes from the actual local HTTP server. The sample below always runs search-and-SQL mode, even if other notebooks have live credentials.

In [ ]:
try:
    ok=request(env['SPEND_TOKEN'],{'question':'Nova Q1','query':{'quarter':'2026-Q1','supplier':'Nova'}})
    denied=request('wrong-token',{'question':'anything'})
    forged=request(env['POLICY_TOKEN'],{'question':'payment terms','tenant':'beacon'})
    assert ok[0]==200 and ok[1]['total_minor']==6500000
    assert denied[0]==401 and forged[0]==400
    print('Spend:',ok,'Invalid credential:',denied,'Tenant override:',forged)
finally:
    process.terminate();process.wait(timeout=5)
    process.stderr.close()

**35–55 min | Map to a client landing zone.** Replace static tokens with IdP-issued identities, validate issuer/audience/expiry, add TLS/ingress, secret management, workload identity, least-privilege data access, network egress controls and audit retention. Choose managed containers, serverless or Kubernetes only from workload and client constraints. Draw runtime, data, identity and management boundaries.

**55–70 min | Threat-review a changed design.** A vendor document contains instructions to reveal finance data. Explain why prompt instructions alone are insufficient, where source approval and ACLs apply, and how tool authority is constrained. Dockerfile is supplied for an optional local image build; Docker, cloud deployment, enterprise IAM and load testing are not demonstrated by this notebook. Record those validation gaps explicitly.